# PhysAR-IDS -- NB15 (v6): figures, one file per figure

Every figure is produced **separately**, as its own single chart with its own file, so it can be dropped into the dissertation one-to-one. Nothing is fitted: the notebook only reads the `*_v6.csv` / `*_v6.pkl` result files, so it runs in about a minute and can be rerun whenever a notebook is rerun.

**Output folder** `figures_v6/`:
* `<ID>_v6.png` (300 dpi, for Word) and `<ID>_v6.pdf` (vector, for LaTeX) for every figure, no title inside the image (the dissertation caption carries it);
* `figure_index_v6.csv`: ID, file, the v4 figure it replaces or the section it belongs in, caption, source tables, status;
* `figure_captions_v6.md`: the same captions as plain text, ready to paste under each figure.

**IDs.** `F6.x` = replaces Figure 6.x of the v4 dissertation (multi-panel v4 figures are split: `F6.6a` ... `F6.6d`). `N-...` = figures for the v6 results the v4 text does not have (feature ablation, defence-component ablation, conformal prediction, complexity, tuning, gate calibration, ESI / PCR, SHAP, MSU, split sensitivity).

**Conventions on every figure.** Grey-box = PGD on the NB3 surrogate; black-box = PPO with a hard-label oracle; white-box = tree-greedy search on the model's own trees (a lower bound on a real white-box attacker). Evasion = share of attack rows classified benign after the attack, **including the roughly 1 % already missed on clean data**. Error bars = std over the 5 seeds, which measures model variance only (the test split holds 422 independent bursts). A figure whose source file is missing is skipped and listed as `MISSING` in the index.

## 0. Setup and helpers

In [ ]:
import os, re, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.chdir(PROJECT_DIR)
PIPELINE_VERSION = 'v6'; V = PIPELINE_VERSION
SMOKE_TEST = False
DPI = 300
FIGDIR = f'figures_{V}'; os.makedirs(FIGDIR, exist_ok=True)
OI = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'verm': '#D55E00', 'sky': '#56B4E9', 'purple': '#CC79A7', 'grey': '#6b6b6b', 'black': '#222222'}
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10, 'legend.frameon': False, 'savefig.bbox': 'tight', 'pdf.fonttype': 42})
CL = ['backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']; CL6 = ['benign'] + CL
print('Working directory:', os.getcwd(), '| figures go to', FIGDIR)

def _find(fname):
    for d in [os.getcwd(), BASE_DIR, os.path.dirname(BASE_DIR)]:
        p = os.path.join(d, fname)
        if os.path.exists(p): return p
    raise FileNotFoundError(f'{fname} not found in {os.getcwd()}, {BASE_DIR} or its parent')
def rd(stem): return pd.read_csv(_find(f'{stem}_{V}.csv'))
def num(s):
    m = re.search(r'[-+]?\d+\.?\d*(?:[eE][-+]?\d+)?', str(s)); return float(m.group()) if m else np.nan
def sdv(s):
    m = re.search(r'\+/-\s*([\d.]+)', str(s)); return float(m.group(1)) if m else 0.0
def arrow(s):
    t = re.findall(r'[-+]?\d+\.\d+|[-+]?\d+', str(s).split('(')[0]); return (float(t[0]), float(t[1])) if len(t) >= 2 else (np.nan, np.nan)
def row(df, col, pat, exact=False):
    m = df[col].astype(str).eq(pat) if exact else df[col].astype(str).str.contains(pat, regex=False)
    if not m.any(): raise KeyError(f'{pat!r} not found in column {col!r}')
    return df[m].iloc[0]
def patch(c): return plt.Rectangle((0, 0), 1, 1, color=c)

INDEX = []
def figure(fid, place, caption, sources, size=(7.5, 4.3)):
    # decorator: the function draws on a fresh (fig, ax) and returns nothing; the figure is saved on its own as <fid>_v6.png / .pdf
    def deco(fn):
        t0 = time.time(); stem = fid.replace('.', '_')
        try:
            fig, ax = plt.subplots(figsize=size); fn(fig, ax); fig.tight_layout()
            fig.savefig(os.path.join(FIGDIR, f'{stem}_{V}.png'), dpi=DPI); fig.savefig(os.path.join(FIGDIR, f'{stem}_{V}.pdf')); plt.show(); plt.close(fig)
            INDEX.append({'id': fid, 'file': f'{stem}_{V}.png', 'place in dissertation': place, 'caption': caption, 'sources': ', '.join(sources), 'status': 'ok'}); print(f'{fid:10s} OK  ({time.time() - t0:.1f}s)')
        except (FileNotFoundError, KeyError, ValueError, IndexError) as e:
            plt.close('all'); INDEX.append({'id': fid, 'file': '', 'place in dissertation': place, 'caption': caption, 'sources': ', '.join(sources), 'status': f'MISSING: {e}'}); print(f'{fid:10s} MISSING -> {e}')
        return fn
    return deco
def bars(ax, labels, series, colors, names=None, err=None, width=0.82, fmt='{:.2f}', ymax=None, rot=90):
    n = len(series); w = width / n; x = np.arange(len(labels))
    for i, s in enumerate(series):
        e = None if err is None else np.nan_to_num(np.asarray(err[i], float)); s = np.asarray(s, float)
        ax.bar(x + (i - (n - 1) / 2) * w, np.nan_to_num(s), w, yerr=e, capsize=2, color=colors[i], error_kw={'lw': 1})
        for xi, v, e_ in zip(x + (i - (n - 1) / 2) * w, s, np.zeros(len(s)) if e is None else e):
            if np.isfinite(v): ax.text(xi, v + e_ + 0.01, fmt.format(v), ha='center', va='bottom', fontsize=6.5, rotation=rot)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    if names: ax.legend(handles=[patch(c) for c in colors], labels=names, fontsize=8, ncol=min(3, n), loc='upper left')
    if ymax: ax.set_ylim(0, ymax)

## 1. Attack figures (v4 Figures 6.2 to 6.5)

In [ ]:
def _sweep(ax, which, ylabel):
    d = rd('attack1_sweep'); eps = d['eps'].astype(float).values
    for col, c, lab in [('FGSM', OI['sky'], 'FGSM'), ('PGD', OI['blue'], 'PGD (10 steps)')]:
        key = f'{col} {which}'; m = d[key].map(num).values; s = d[key].map(sdv).values
        ax.plot(eps, m, marker='o', ms=4, color=c, label=lab); ax.fill_between(eps, m - s, m + s, color=c, alpha=0.18)
    ax.set_xlabel('perturbation bound epsilon (sigma, L-inf)'); ax.set_ylabel(ylabel); ax.set_ylim(0, 1); ax.legend(loc='upper right' if which == 'F1' else 'center right')
    return d
@figure('F6.2a', 'Replaces Figure 6.2 (left panel), section 6.2', 'Weighted F1 of the tuned XGBoost classifier under grey-box FGSM and PGD attacks (surrogate from NB3) as the perturbation bound epsilon grows. Mean and std over 5 seeds.', ['attack1_sweep'])
def f62a(fig, ax): _sweep(ax, 'F1', 'weighted F1')
@figure('F6.2b', 'Replaces Figure 6.2 (right panel), section 6.2', 'Evasion rate (share of attack rows classified benign) under grey-box FGSM and PGD as epsilon grows. The dashed line is the share already missed on clean data (epsilon = 0). Mean and std over 5 seeds.', ['attack1_sweep'])
def f62b(fig, ax):
    d = _sweep(ax, 'Evasion', 'evasion'); b = num(d['PGD Evasion'].iloc[0]); ax.axhline(b, ls='--', color=OI['grey'], lw=1); ax.text(0.12, b + 0.015, f'clean miss {b:.3f}', ha='left', fontsize=8, color=OI['grey'])

@figure('F6.3', 'Replaces Figure 6.3, section 6.2', 'Per-class evasion at 0.1 sigma on the 60 cyber features: grey-box PGD against the white-box tree-greedy attack (a lower bound on a white-box attacker). The dashed line is the share of attack rows already missed on clean data. Mean and std over seeds.', ['attack2_evasion_table', 'attack1w_results_table', 'attack2e_f1_table'])
def f63(fig, ax):
    a2 = rd('attack2_evasion_table'); pg = row(a2, 'method', 'PGD_cyber_eps0.1', exact=True); w = rd('attack1w_results_table'); wb = row(w, 'setting', 'cyber_0.1', exact=True)
    cm = rd('attack2e_f1_table'); cm = cm[cm['method'] == 'clean']; base = 1 - num(cm['attack rows labelled non-benign'].iloc[0]) if len(cm) else np.nan
    labs = CL + ['all attacks']; k = CL + ['ALL']
    bars(ax, labs, [[num(pg[c]) for c in k], [num(wb[c]) for c in k]], [OI['blue'], OI['verm']], ['grey-box PGD, 0.1 sigma', 'white-box tree-greedy, 0.1 sigma'], err=[[sdv(pg[c]) for c in k], [sdv(wb[c]) for c in k]], fmt='{:.3f}', ymax=1.2)
    if np.isfinite(base): ax.axhline(base, ls='--', color=OI['grey'], lw=1); ax.text(2.55, base + 0.015, f'clean miss {base:.3f}', fontsize=8, color=OI['grey'])
    ax.set_ylabel('evasion')

@figure('F6.4', 'Replaces Figure 6.4, section 6.3', 'Evasion by attacker and class against the tuned XGBoost classifier: grey-box PGD, random perturbation and black-box PPO (hard-label oracle, 10 queries per sample), each at a matched budget and unconstrained. Mean and std over seeds.', ['attack2_evasion_table'], size=(10, 4.6))
def f64(fig, ax):
    a2 = rd('attack2_evasion_table'); order = [('PGD_cyber_eps0.1', 'PGD grey-box 0.1', OI['blue']), ('Random_budgeted', 'random, budgeted', OI['sky']), ('PPO_budgeted', 'PPO black-box, budgeted', OI['green']), ('Random_unconstrained', 'random, unconstrained', OI['orange']), ('PPO_unconstrained', 'PPO black-box, unconstrained', OI['verm'])]
    labs = CL + ['ALL']; ser, err, col, nm = [], [], [], []
    for key, lab, c in order:
        r = row(a2, 'method', key, exact=True); ser.append([num(r[x]) for x in labs]); err.append([sdv(r[x]) for x in labs]); col.append(c); nm.append(lab)
    bars(ax, labs[:-1] + ['all attacks'], ser, col, nm, err=err, width=0.86, ymax=1.2); ax.set_ylabel('evasion')

@figure('F6.5', 'Replaces Figure 6.5, section 6.3', 'Mean number of queries (maximum 10) the black-box PPO agent needs before the first misclassification, per class, over evaded rows only. The percentage under each bar is the share of attack rows that evade; no step count is shown where almost none do. Mean and std over seeds.', ['attack2_results'])
def f65(fig, ax):
    d = joblib.load(f'attack2_results_{V}.pkl'); seeds = list(d['per_seed']); w = 0.38
    for i, (m, lab, c) in enumerate([('PPO_budgeted', 'PPO budgeted', OI['green']), ('PPO_unconstrained', 'PPO unconstrained', OI['verm'])]):
        st = np.array([d['per_seed'][s][m]['mean_steps_to_evade'] for s in seeds], float); ev = np.array([d['per_seed'][s][m]['per_class_evasion'] for s in seeds], float)
        mu = np.nanmean(st, 0); sd_ = np.nanstd(st, 0, ddof=1) if len(seeds) > 1 else np.zeros(5); x = np.arange(5) + (i - 0.5) * w
        ax.bar(x, np.nan_to_num(mu), w, yerr=np.nan_to_num(sd_), capsize=2, color=c, label=lab)
        for xi, mi, e in zip(x, mu, ev.mean(0)): ax.text(xi, (0 if np.isnan(mi) else mi) + 0.15, 'none' if (np.isnan(mi) or e < 0.01) else f'{mi:.1f}\n{e:.0%}', ha='center', fontsize=7)
    ax.set_xticks(range(5)); ax.set_xticklabels(CL); ax.set_ylabel('mean queries to evade'); ax.set_ylim(0, 6); ax.legend(loc='upper left')

## 2. Defence figures (v4 Figures 6.6 to 6.10)

In [ ]:
_D1 = [('PGD, NB3 surrogate', 'a', 'grey-box PGD (attacker unseen in training)'), ('PPO 0.3 sigma', 'b', 'black-box PPO, 0.3 sigma (budget seen in training)'),
       ('PPO unconstrained', 'c', 'black-box PPO, unconstrained (budget unseen)'), ('white-box tree-greedy 0.3', 'd', 'white-box tree-greedy, 0.3 sigma (adaptive)')]
for pat, letter, lab in _D1:
    def _mk(pat=pat, lab=lab):
        def f(fig, ax):
            d = rd('defence1_per_class_table'); r = row(d, 'attack', pat); a = [arrow(r[c]) for c in CL]
            bars(ax, CL, [[x[0] for x in a], [x[1] for x in a]], [OI['grey'], OI['blue']], ['standard (tuned) model', 'adversarially trained (PGD + PPO examples, NB5)'], fmt='{:.2f}', ymax=1.25); ax.set_ylabel('evasion'); ax.set_title(lab, fontsize=9, loc='left', color=OI['grey'])
        return f
    figure(f'F6.6{letter}', 'Replaces Figure 6.6, section 6.4 (one panel per attacker)', f'Per-class evasion before and after adversarial training on PGD and PPO examples (NB5), against {lab}. Mean over seeds.', ['defence1_per_class_table'])(_mk())

def _a3rows():
    a = rd('attack3_per_class'); a = a[a['Layer 1'] == 'rob']; rows_ = [(r['budget'], [arrow(r[c]) for c in CL], 'NB6') for _, r in a.iterrows()]
    try:
        b = rd('attack3b_per_class'); b = b[b['Layer 1'] == 'rob']; rows_ += [(r['budget'], [arrow(r[c]) for c in CL], 'NB6b converged') for _, r in b.iterrows()]
    except FileNotFoundError: pass
    keep = [t for t in rows_ if not (t[2] == 'NB6' and any(num(u[0]) == num(t[0]) and u[2] != 'NB6' for u in rows_))]
    return sorted(keep, key=lambda t: num(t[0]))
for k, letter in enumerate('abcd'):
    def _mk(k=k):
        def f(fig, ax):
            rows_ = _a3rows(); bud, ar, src = rows_[k]
            bars(ax, CL, [[x[0] for x in ar], [x[1] for x in ar]], [OI['sky'], OI['verm']], ['attacker ignores the gate', 'fully adaptive (attacks classifier and gate)'], fmt='{:.2f}', ymax=1.25)
            ax.set_ylabel('evasion of the pipeline (robust model + gate)'); ax.set_title(f'white-box {bud} [{src}]', fontsize=9, loc='left', color=OI['grey'])
        return f
    figure(f'F6.7{letter}', 'Replaces Figure 6.7, section 6.5 (one panel per budget, increasing)', 'Per-class evasion of the pipeline (white-box-trained robust model plus autoencoder gate) against the white-box tree-greedy attack that ignores the gate and against the fully adaptive attack that targets both. One panel per budget; the budget and notebook are printed in the panel. Mean over seeds; a lower bound on a white-box attacker.', ['attack3_per_class', 'attack3b_per_class'])(_mk())

for m, letter, ttl in [('rob', 'a', 'the robust model (iterative white-box adversarial training, NB5b)'), ('def', 'b', 'the model adversarially trained on PGD and PPO examples (NB5)')]:
    def _mk(m=m, ttl=ttl):
        def f(fig, ax):
            t = rd('attack3_table'); tt = t[t['Layer 1'] == m]; buds = [0.1, 0.3, 1.0]
            un = [row(tt, 'attacker', f'white-box {b} sigma, gate-UNAWARE') for b in buds]; ad = [row(tt, 'attacker', f'white-box {b} sigma, FULLY ADAPTIVE') for b in buds]
            for rr, col, lab, mk, c in [(un, 'evades Layer 1', 'classifier alone', 'o', OI['grey']), (un, 'evades pipeline (L1 + gate)', 'classifier + gate, gate ignored by the attacker', 's', OI['blue']), (ad, 'evades pipeline (L1 + gate)', 'classifier + gate, fully adaptive attacker', '^', OI['verm'])]:
                ax.errorbar(buds, [num(r[col]) for r in rr], yerr=[sdv(r[col]) for r in rr], marker=mk, color=c, capsize=3, label=lab)
            ax.set_xscale('log'); ax.set_xticks(buds); ax.set_xticklabels([str(b) for b in buds]); ax.set_xlabel('white-box budget (sigma, L-inf, cyber features)'); ax.set_ylabel('evasion'); ax.set_ylim(0, 1.05); ax.legend(fontsize=8, loc='upper left')
        return f
    figure(f'F6.8{letter}', 'Replaces Figure 6.8, section 6.5', f'Evasion against the white-box tree-greedy attack as the budget grows, for {ttl}: the classifier alone, the classifier with the gate when the attacker ignores it, and the fully adaptive attacker. Mean and std over seeds; a lower bound on a white-box attacker.', ['attack3_table'])(_mk())

def _gate_tables(): return rd('gate7_gate_vs_layer1'), rd('gate7_gate_gradient_attack')
@figure('F6.9a', 'Replaces Figure 6.9 (left), section 6.6', 'Clean test data: share of attack rows detected and share of benign rows falsely alarmed, for each classifier alone, each gate alone and each combination (gate threshold at the 95th percentile of the out-of-fold benign error). Mean and std over seeds.', ['gate7_gate_vs_layer1'], size=(8, 5))
def f69a(fig, ax):
    g, _ = _gate_tables(); names = g['system'].tolist(); y = np.arange(len(names))
    ax.barh(y - 0.2, g['detection of attack rows'].map(num), 0.4, xerr=g['detection of attack rows'].map(sdv), color=OI['blue'], capsize=2, label='attack rows detected')
    ax.barh(y + 0.2, g['false alarms (benign rows)'].map(num), 0.4, xerr=g['false alarms (benign rows)'].map(sdv), color=OI['verm'], capsize=2, label='benign rows falsely alarmed')
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=8); ax.invert_yaxis(); ax.set_xlim(0, 1.05); ax.legend(fontsize=8, loc='lower right')
@figure('F6.9b', 'Replaces Figure 6.9 (middle), section 6.6', 'Clean detection rate per attack class for each classifier, gate and combination. The gate alone sees ransomware and reverseshell but almost never FDI. Mean over seeds.', ['gate7_gate_vs_layer1'], size=(7.5, 5))
def f69b(fig, ax):
    g, _ = _gate_tables(); M = np.array([[num(r[c]) for c in CL] for _, r in g.iterrows()]); ax.grid(False); ax.imshow(M, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]): ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8, color='w' if M[i, j] > 0.6 else 'k')
    ax.set_xticks(range(5)); ax.set_xticklabels(CL, fontsize=8, rotation=20); ax.set_yticks(range(len(g))); ax.set_yticklabels(g['system'], fontsize=8)
@figure('F6.9c', 'Replaces Figure 6.9 (right), section 6.6', 'Share of attack rows still flagged by the gate alone after a white-box gradient attack on the gate (cyber features, 0.3 and 1.0 sigma), for each gate variant. Mean over seeds.', ['gate7_gate_gradient_attack'])
def f69c(fig, ax):
    _, ga = _gate_tables()
    for _, r in ga.iterrows(): ax.plot([0, 0.3, 1.0], [num(r['clean']), num(r['0.3 sigma']), num(r['1.0 sigma'])], marker='o', label=str(r['gate']))
    ax.set_xticks([0, 0.3, 1.0]); ax.set_xticklabels(['clean', '0.3 sigma', '1.0 sigma']); ax.set_ylim(0, 1); ax.set_ylabel('attack rows flagged by the gate alone'); ax.legend(fontsize=8)

@figure('F6.10', 'Replaces Figure 6.10, section 6.6', 'Evasion by attacker tier through the defence stages: standard model, robust model (white-box adversarial training), robust model plus gate when the attacker ignores the gate, and plus gate against the fully adaptive attacker. Evasion includes the share already missed on clean data. Mean and std over seeds. Bars are empty where that attacker or budget was not run.', ['attack2_evasion_table', 'attack2b_sweep_table', 'attack1w_results_table', 'attack3_table'], size=(9, 4.6))
def f610(fig, ax):
    a2 = rd('attack2_evasion_table'); sw = rd('attack2b_sweep_table'); w = rd('attack1w_results_table'); t = rd('attack3_table'); t = t[t['Layer 1'] == 'rob']
    def wbstd(s):
        r = w[w['setting'] == s]; return (num(r['ALL'].iloc[0]), sdv(r['ALL'].iloc[0])) if len(r) else (np.nan, 0)
    ppo03 = sw[(sw['method'] == 'PPO') & (sw['budget'].astype(float) == 0.3)]
    tiers = [('grey-box PGD\n0.1 sigma', 'PGD_cyber_eps0.1', 'PGD, NB3', None), ('black-box PPO\n0.3 sigma', 'PPO0.3', 'PPO 0.3 sigma, trained vs standard', 'PPO 0.3 sigma, FULLY ADAPTIVE'),
             ('white-box\n0.3 sigma', 'cyber_0.3', 'white-box 0.3 sigma, gate-UNAWARE', 'white-box 0.3 sigma, FULLY ADAPTIVE'), ('white-box\n1.0 sigma', 'cyber_1.0', 'white-box 1.0 sigma, gate-UNAWARE', 'white-box 1.0 sigma, FULLY ADAPTIVE')]
    S = {k: [] for k in ['std', 'rob', 'gate', 'adapt']}; E = {k: [] for k in S}
    for lab, std_key, un_pat, ad_pat in tiers:
        if std_key == 'PGD_cyber_eps0.1': r = row(a2, 'method', std_key, exact=True); s = (num(r['ALL']), sdv(r['ALL']))
        elif std_key == 'PPO0.3': s = (num(ppo03['ALL'].iloc[0]), sdv(ppo03['ALL'].iloc[0])) if len(ppo03) else (np.nan, 0)
        else: s = wbstd(std_key)
        ru = row(t, 'attacker', un_pat); S['std'].append(s[0]); E['std'].append(s[1]); S['rob'].append(num(ru['evades Layer 1'])); E['rob'].append(sdv(ru['evades Layer 1']))
        S['gate'].append(num(ru['evades pipeline (L1 + gate)'])); E['gate'].append(sdv(ru['evades pipeline (L1 + gate)']))
        if ad_pat is None: S['adapt'].append(np.nan); E['adapt'].append(0)
        else: ra = row(t, 'attacker', ad_pat); S['adapt'].append(num(ra['evades pipeline (L1 + gate)'])); E['adapt'].append(sdv(ra['evades pipeline (L1 + gate)']))
    ks = ['std', 'rob', 'gate', 'adapt']
    bars(ax, [x[0] for x in tiers], [S[k] for k in ks], [OI['grey'], OI['blue'], OI['green'], OI['verm']], ['standard model', '+ white-box adversarial training (robust, NB5b)', '+ gate (attacker ignores the gate)', '+ gate, fully adaptive attacker'], err=[E[k] for k in ks], width=0.8, ymax=1.3); ax.set_ylabel('evasion')

## 3. Federated learning (v4 Figure 6.16)

In [ ]:
def _flcol(s):
    s = s.lower()
    if 'centralised' in s: return OI['black']
    if 'local-only' in s: return OI['grey']
    if 'byzantine' in s: return OI['verm']
    if 'dp-fl' in s: return OI['purple']
    if 'physfed' in s or 'trimmed' in s or 'uniform' in s: return OI['orange']
    return OI['blue']
@figure('F6.16a', 'Replaces Figure 6.16 (left), section 6.13', 'Weighted F1 of the federated and centralised 93-feature MLPs on the temporal test split (9 clients). Colours: black = centralised, grey = local only, blue = FedAvg / FedProx, orange = weighting and trimmed-mean aggregators, red = with one Byzantine client, purple = differentially private. The dashed line is the 150-epoch centralised reference. Mean and std over seeds.', ['fl_main'], size=(8, 6))
def f616a(fig, ax):
    m = rd('fl_main'); n = len(m); y = np.arange(n)[::-1]; f1 = m['weighted F1'].map(num).values; e1 = m['weighted F1'].map(sdv).values
    ax.barh(y, f1, xerr=e1, color=[_flcol(c) for c in m['configuration']], capsize=2); ax.axvline(f1[0], ls='--', color=OI['black'], lw=1)
    for yi, v, e_ in zip(y, f1, e1): ax.text(v + e_ + 0.012, yi, f'{v:.3f}', va='center', fontsize=7.5)
    ax.set_yticks(y); ax.set_yticklabels(m['configuration'], fontsize=8); ax.set_xlabel('weighted F1'); ax.set_xlim(0, 0.9)
@figure('F6.16b', 'Replaces Figure 6.16 (right), section 6.13', 'Evasion of attack rows for each federated configuration, clean (blue) and after a white-box PGD attack on the global model (red). Mean over seeds.', ['fl_attack'], size=(8, 6))
def f616b(fig, ax):
    a = rd('fl_attack'); n = len(a); y = np.arange(n)[::-1]; ec = a['evasion clean'].map(num).values; ep = a['evasion PGD'].map(num).values
    for yi, c_, p_ in zip(y, ec, ep): ax.plot([c_, p_], [yi, yi], color=OI['grey'], lw=1.5)
    ax.scatter(ec, y, color=OI['sky'], s=40, zorder=3, label='clean'); ax.scatter(ep, y, color=OI['verm'], s=40, zorder=3, label='after white-box PGD')
    ax.set_yticks(y); ax.set_yticklabels(a['configuration'], fontsize=8); ax.set_xlabel('evasion'); ax.legend(fontsize=8)

## 4. New v6 results: baselines, tuning, splits, features

In [ ]:
@figure('N-BASE1', 'Results: baseline models (section 6.1)', 'Weighted F1 of every baseline model on the temporal test split (untuned, same hyper-parameters as NB1, 5 seeds). Mean and std over seeds.', ['nb1_main_table_v6_multiseed'], size=(7.5, 4.5))
def nbase1(fig, ax):
    d = pd.read_csv(f'nb1_main_table_{V}_multiseed.csv'); d = d.assign(f=d['Test F1 (weighted)'].map(num), e=d['Test F1 (weighted)'].map(sdv)).sort_values('f'); y = np.arange(len(d))
    ax.barh(y, d['f'], xerr=d['e'], color=OI['blue'], capsize=2)
    for yi, v, e_ in zip(y, d['f'], d['e']): ax.text(v + e_ + 0.01, yi, f'{v:.3f}', va='center', fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(d['Model']); ax.set_xlabel('weighted F1'); ax.set_xlim(0, 1)
@figure('N-OPT1', 'Results: hyper-parameter tuning (section 6.1)', 'Optuna TPE search for XGBoost: cross-validated weighted F1 of every completed trial (dots) and the best value so far (line). Selection used time-blocked cross-validation inside the training data only.', ['optuna_xgb_trials'])
def nopt1(fig, ax):
    d = pd.read_csv(f'optuna_xgb_trials_{V}.csv'); d = d[d['state'] == 'COMPLETE']; ax.scatter(d['number'], d['value'], s=14, color=OI['sky']); ax.plot(d['number'], d['value'].cummax(), color=OI['blue'], lw=2, label='best so far')
    ax.set_xlabel('trial'); ax.set_ylabel('cross-validated weighted F1'); ax.legend()
@figure('N-TUNE1', 'Results: tuned vs untuned, other models (section 6.1)', 'Weighted F1 of each model with default and with Optuna-tuned hyper-parameters, refitted in the same session on the same 5 seeds. Mean and std over seeds.', ['nb2a_summary'], size=(8.5, 4.3))
def ntune1(fig, ax):
    d = rd('nb2a_summary'); bars(ax, d['Model'].tolist(), [d['Baseline (here)'].map(num), d['Tuned'].map(num)], [OI['grey'], OI['blue']], ['default', 'tuned'], err=[d['Baseline (here)'].map(sdv), d['Tuned'].map(sdv)], ymax=1.05); ax.set_ylabel('weighted F1'); ax.tick_params(axis='x', labelrotation=25)
@figure('N-SPLIT1', 'Results: split sensitivity (section 6.1 or limitations)', 'Weighted F1 of XGBoost on all, physical-only and cyber-only features under different temporal split choices. Mean and std over seeds.', ['split_sensitivity_summary'], size=(8, 4.8))
def nsplit1(fig, ax):
    d = rd('split_sensitivity_summary'); y = np.arange(len(d))[::-1]
    for off, col, c, lab in [(-0.2, 'XGBoost|all', OI['blue'], 'all features'), (0, 'XGBoost|cyber_only', OI['green'], 'cyber only'), (0.2, 'XGBoost|physical_only', OI['verm'], 'physical only')]:
        ax.errorbar(d[col].map(num), y + off, xerr=d[col].map(sdv), fmt='o', color=c, capsize=2, label=lab)
    ax.set_yticks(y); ax.set_yticklabels(d['split'], fontsize=8); ax.set_xlabel('weighted F1'); ax.legend(fontsize=8)
@figure('N-SHAP1', 'Results: explainability (section 6.8)', 'Share of mean absolute TreeSHAP attribution per class by feature group (raw physical, derived physical, temporal physical, cyber at 1 s, 2 s and 5 s windows). Mean over seeds.', ['shap_group_share'], size=(8, 4.3))
def nshap1(fig, ax):
    d = rd('shap_group_share'); cols = ['raw physical', 'derived physical', 'temporal physical', 'cyber 1 s', 'cyber 2 s', 'cyber 5 s']; pal = [OI['verm'], OI['orange'], OI['purple'], OI['sky'], OI['blue'], OI['green']]; left = np.zeros(len(d)); y = np.arange(len(d))[::-1]
    for c, col in zip(cols, pal): ax.barh(y, d[c].astype(float), left=left, color=col, label=c); left += d[c].astype(float).values
    ax.set_yticks(y); ax.set_yticklabels(d['class']); ax.set_xlabel('share of attribution'); ax.legend(fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.15))
@figure('N-SHAP2', 'Results: explainability (section 6.8)', 'Weighted F1 of XGBoost refitted on feature subsets chosen from the SHAP analysis. Mean and std over seeds.', ['shap_feature_subset_f1'], size=(8, 4))
def nshap2(fig, ax):
    d = rd('shap_feature_subset_f1'); y = np.arange(len(d))[::-1]; ax.barh(y, d['weighted F1'].map(num), xerr=d['weighted F1'].map(sdv), color=OI['blue'], capsize=2)
    for yi, v in zip(y, d['weighted F1'].map(num)): ax.text(v + 0.01, yi, f'{v:.3f}', va='center', fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(d['feature set']); ax.set_xlabel('weighted F1'); ax.set_xlim(0, 1)
@figure('N-PCR1', 'Results: ESI and PCR (section 6.9)', 'Physical-Channel Reliance (PCR) per class for the standard and robust models on clean data: the share of gradient-times-input attribution carried by the physical channels. Mean and std over seeds.', ['esipcr_pcr_clean'], size=(8.5, 4.3))
def npcr1(fig, ax):
    d = rd('esipcr_pcr_clean'); d = d[~d['model'].astype(str).str.contains('PCR-derived', regex=False)]; cols = [OI['blue'], OI['verm'], OI['green'], OI['orange']]
    bars(ax, CL6, [[num(r[c]) for c in CL6] for _, r in d.iterrows()], cols[:len(d)], d['model'].tolist(), err=[[sdv(r[c]) for c in CL6] for _, r in d.iterrows()], ymax=0.5); ax.set_ylabel('PCR')
@figure('N-ESI1', 'Results: ESI and PCR (section 6.9)', 'Evasion-Success Index (ESI) per attack class for the standard model under each attack (bars) with the matched random-perturbation control (markers). Mean over seeds.', ['esipcr_esi_per_class'], size=(8.5, 4.3))
def nesi1(fig, ax):
    d = rd('esipcr_esi_per_class'); d = d[d['model'].astype(str).str.startswith('std')]; cols = [OI['blue'], OI['verm'], OI['green'], OI['orange']]
    vals = [[num(r[c]) for c in CL] for _, r in d.iterrows()]; ctrl = [[float(re.search(r'ctrl\s*([-\d.]+)', str(r[c])).group(1)) for c in CL] for _, r in d.iterrows()]; n = len(d); w = 0.8 / n; x = np.arange(5)
    for i in range(n):
        ax.bar(x + (i - (n - 1) / 2) * w, vals[i], w, color=cols[i % 4], label=str(d.iloc[i]['attack'])); ax.scatter(x + (i - (n - 1) / 2) * w, ctrl[i], marker='_', s=120, color=OI['black'], zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(CL); ax.set_ylabel('ESI (markers: random control)'); ax.legend(fontsize=8, title='attack')

## 5. New v6 results: ablations

In [ ]:
@figure('N-ABL1', 'Results: feature ablation (new section)', 'Clean weighted F1 of XGBoost refitted on nested feature sets (tuned parameters, 5 seeds). The 93-feature model is the reference; dropping the four derived physical features changes the F1 by less than one standard deviation. Mean and std over seeds.', ['ablation_A_clean'], size=(8, 4.3))
def nabl1(fig, ax):
    d = rd('ablation_A_clean'); bars(ax, d['feature set'].tolist(), [d['weighted F1'].map(num)], [OI['blue']], err=[d['weighted F1'].map(sdv)], fmt='{:.3f}', rot=0, ymax=1.05); ax.set_ylabel('weighted F1'); ax.tick_params(axis='x', labelrotation=25)
@figure('N-ABL2', 'Results: feature ablation (new section)', 'Per-class recall of XGBoost on each feature set (clean test split, mean over seeds). Physical channels alone fail on benign data; they add most for ransomware and reverseshell.', ['ablation_A_recall_pc'], size=(7.5, 4.3))
def nabl2(fig, ax):
    d = rd('ablation_A_recall_pc'); M = d[CL6].astype(float).values; ax.grid(False); ax.imshow(M, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]): ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8, color='w' if M[i, j] > 0.6 else 'k')
    ax.set_xticks(range(6)); ax.set_xticklabels(CL6, fontsize=8, rotation=20); ax.set_yticks(range(len(d))); ax.set_yticklabels(d['feature set'])
@figure('N-ABL3', 'Results: feature ablation (new section)', 'Evasion after the white-box tree-greedy attack run against each feature set\'s own trees: cyber attacker at 0.3 and 1.0 sigma, and an attacker that may also move physical channels (0.3 sigma). A set with no cyber feature has no cyber-attacker result. Mean and std over seeds.', ['ablation_A_attack'], size=(9.5, 4.6))
def nabl3(fig, ax):
    d = rd('ablation_A_attack'); sets = list(dict.fromkeys(d['feature set'])); series = []; names = []; cols = [OI['orange'], OI['verm'], OI['green']]
    for (atk, b), c in zip([('cyber', 0.3), ('cyber', 1.0), ('any', 0.3)], cols):
        sub = d[(d['attacker'] == atk) & (d['budget (sigma)'].astype(float) == b)].set_index('feature set'); series.append((c, f'{atk} attacker, {b:g} sigma', [num(sub.loc[s, 'evasion after attack']) if s in sub.index else np.nan for s in sets], [sdv(sub.loc[s, 'evasion after attack']) if s in sub.index else 0 for s in sets]))
    bars(ax, sets, [s[2] for s in series], [s[0] for s in series], [s[1] for s in series], err=[s[3] for s in series], ymax=1.3); ax.set_ylabel('evasion'); ax.tick_params(axis='x', labelrotation=25)
_CLS = {'base': 'tuned (standard)', 'rob': 'white-box adv. trained (NB5b)', 'dfn': 'PGD/PPO adv. trained (NB5)'}
@figure('N-ABL4', 'Results: defence-component ablation (new section)', 'Evasion of attack rows by classifier (standard, white-box adversarially trained, PGD/PPO adversarially trained), alone and combined with the autoencoder gate, under clean data, grey-box PGD and white-box attacks that target the classifier only. Mean and std over seeds.', ['ablation_B_evasion'], size=(10, 4.8))
def nabl4(fig, ax):
    d = rd('ablation_B_evasion'); conds = list(dict.fromkeys(d['condition'])); series, cols, names = [], [OI['grey'], OI['black'], OI['blue'], OI['sky'], OI['green'], OI['orange']], []; errs = []
    for m in ['base', 'rob', 'dfn']:
        for key, tag in [('cls alone', 'classifier'), ('both', '+ gate')]:
            sub = d[d['classifier'] == m].set_index('condition'); series.append([num(sub.loc[c, key]) for c in conds]); errs.append([sdv(sub.loc[c, key]) for c in conds]); names.append(f'{_CLS[m]}, {tag}')
    bars(ax, [{'clean': 'clean', 'pgd': 'grey-box PGD', 'wb0.3': 'white-box 0.3 sigma', 'wb1': 'white-box 1.0 sigma'}.get(c, c) for c in conds], series, cols, names, err=errs, width=0.86, rot=90, ymax=1.35); ax.set_ylabel('evasion'); 
@figure('N-ABL5', 'Results: defence-component ablation (new section)', 'Cost and benefit of each defence combination: benign false-alarm rate on clean data against evasion under the 1.0 sigma white-box attack (classifier alone, or classifier OR gate). Mean over seeds.', ['ablation_B_clean', 'ablation_B_evasion'], size=(7, 4.8))
def nabl5(fig, ax):
    c = rd('ablation_B_clean').set_index('classifier'); e = rd('ablation_B_evasion'); e = e[e['condition'] == e['condition'].iloc[-1]].set_index('classifier')
    for m, mk in zip(['base', 'rob', 'dfn'], ['o', 's', '^']):
        for fa, ev, tag, col in [('false alarm: classifier', 'cls alone', 'classifier alone', OI['blue']), ('false alarm: classifier OR gate', 'both', '+ gate', OI['verm'])]:
            ax.scatter(num(c.loc[m, fa]), num(e.loc[m, ev]), marker=mk, s=80, color=col, label=f'{_CLS[m]}, {tag}')
    ax.set_xlabel('benign false-alarm rate (clean test)'); ax.set_ylabel('evasion under 1.0 sigma white-box'); ax.legend(fontsize=7, loc='center right')
@figure('N-GATE1', 'Results: gate calibration (section 6.6)', 'Observed false-alarm rate on benign test rows against the nominal rate implied by the percentile threshold, for each gate variant. Points above the diagonal mean the threshold is too permissive in practice. Mean over seeds.', ['gate7_calibration_fpr'])
def ngate1(fig, ax):
    d = rd('gate7_calibration_fpr'); cols = [c for c in d.columns if c.startswith('nominal')]; nom = np.array([float(re.search(r'([\d.]+)%', c).group(1)) / 100 for c in cols])
    for _, r in d.iterrows(): ax.plot(nom, [max(num(r[c]), 3e-4) for c in cols], marker='o', ms=4, label=str(r['gate']))
    ax.plot([min(nom), max(nom)], [min(nom), max(nom)], 'k:', lw=1); ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('nominal false-alarm rate'); ax.set_ylabel('observed false-alarm rate (zero plotted at 3e-4)'); ax.legend(fontsize=7)
@figure('N-GATE2', 'Results: gate threshold trade-off (section 6.6)', 'Robust model plus gate at several thresholds: evasion at 1.0 sigma against an attacker that ignores the gate (squares) and against the fully adaptive attacker (triangles), as a function of the gate\'s benign false-alarm rate. Mean over seeds.', ['attack3c_tradeoff_rob_1.0'])
def ngate2(fig, ax):
    d = pd.read_csv(f'attack3c_tradeoff_rob_1.0_{V}.csv'); x = d['gate FPR'].map(num)
    ax.scatter(x, d['evades pipeline, gate-unaware'].map(num), marker='s', color=OI['blue'], label='attacker ignores the gate'); ax.scatter(x, d['evades pipeline, FULLY ADAPTIVE'].map(num), marker='^', color=OI['verm'], label='fully adaptive attacker')
    for xi, yi, t in zip(x, d['evades pipeline, FULLY ADAPTIVE'].map(num), d['gate threshold']): ax.annotate(str(t), (xi, yi), textcoords='offset points', xytext=(4, 5), fontsize=6.5)
    ax.set_xlabel('gate benign false-alarm rate'); ax.set_ylabel('pipeline evasion (white-box, 1.0 sigma)'); ax.set_ylim(0, 1.1); ax.legend(fontsize=8, loc='lower right')

## 6. New v6 results: conformal prediction, complexity, cross-dataset

In [ ]:
@figure('N-CONF1', 'Results: conformal prediction (new section)', 'Marginal coverage of the prediction sets on the temporal test split against the nominal level 1 - alpha: split conformal calibrated on the last 15 % of training time (NB11), blocked cross-conformal CV+ (NB11b), its class-conditional variant, and a random-fold control. Mean and std over seeds; the dotted line is exact coverage.', ['conformal_coverage', 'conformalfix_coverage'])
def nconf1(fig, ax):
    a = rd('conformal_coverage'); b = rd('conformalfix_coverage'); series = []
    s = a[a['method'] == 'lac'].sort_values('alpha'); series.append(('split LAC (NB11)', s['nominal'].astype(float).values, s['coverage'].map(num).values, s['coverage'].map(sdv).values, OI['grey']))
    for (f, m), lab, c in [(('blocked', 'cvplus'), 'blocked CV+ (NB11b)', OI['blue']), (('blocked', 'cvplus_cc'), 'blocked CV+, class-conditional', OI['verm']), (('random', 'cvplus'), 'random-fold CV+ (control)', OI['green'])]:
        s = b[(b['folds'] == f) & (b['method'] == m)].sort_values('alpha'); series.append((lab, s['nominal'].astype(float).values, s['coverage'].map(num).values, s['coverage'].map(sdv).values, c))
    for lab, x, y, e, c in series: ax.errorbar(x, y, yerr=e, marker='o', capsize=3, color=c, label=lab)
    ax.plot([0.9, 0.99], [0.9, 0.99], 'k:', lw=1); ax.set_xlabel('nominal coverage 1 - alpha'); ax.set_ylabel('empirical coverage'); ax.legend(fontsize=8)
@figure('N-CONF2', 'Results: conformal prediction (new section)', 'Coverage of true benign rows at each nominal level for split conformal (NB11 recipe reproduced in NB11b, calibration on the last time block) and blocked CV+. The recent benign behaviour is missing from the split recipe\'s model, which is why its benign coverage falls short. Mean and std over seeds.', ['conformalfix_coverage'])
def nconf2(fig, ax):
    b = rd('conformalfix_coverage'); al = sorted(b['alpha'].unique()); names = [('split_last', 'split (last block)', OI['grey']), ('cvplus', 'blocked CV+', OI['blue']), ('cvplus_cc', 'blocked CV+, class-conditional', OI['verm'])]; ser, er, cols, nm = [], [], [], []
    for m, lab, c in names:
        s = b[(b['folds'] == 'blocked') & (b['method'] == m)].set_index('alpha'); ser.append([num(s.loc[a_, 'benign coverage']) for a_ in al]); er.append([sdv(s.loc[a_, 'benign coverage']) for a_ in al]); cols.append(c); nm.append(lab)
    bars(ax, [f'alpha {a_:g}' for a_ in al], ser, cols, nm, err=er, fmt='{:.2f}', rot=0, ymax=1.3); ax.set_ylabel('benign-row coverage')
@figure('N-CONF3', 'Results: conformal prediction under attack (new section)', 'Marginal coverage of the split-conformal LAC sets (alpha 0.1) on the attacked rows compared with the same rows clean, for grey-box PGD and the white-box attack. Coverage falls because the sets are calibrated on clean data. Mean and std over seeds.', ['conformal_attack'])
def nconf3(fig, ax):
    d = rd('conformal_attack'); d = d[(d['method'] == 'lac') & (d['alpha'].astype(float) == 0.1)]
    bars(ax, [{'pgd': 'grey-box PGD', 'wb': 'white-box'}.get(a, a) for a in d['attack']], [d['coverage clean'].map(num), d['coverage attacked'].map(num)], [OI['grey'], OI['verm']], ['clean', 'attacked'], err=[d['coverage clean'].map(sdv), d['coverage attacked'].map(sdv)], rot=0, ymax=1.3); ax.set_ylabel('coverage'); ax.axhline(0.9, ls=':', color='k', lw=1)
@figure('N-CX1', 'Results: computational complexity (section 6.12)', 'P99 single-row inference latency of each pipeline component on the evaluation machine (a desktop-class CPU, not an edge device), against the Tier 1 (10 ms) and Tier 2 (100 ms) bounds. Mean and std over seeds.', ['cx_tier'], size=(8.5, 5))
def ncx1(fig, ax):
    d = rd('cx_tier'); d = d.assign(p=d['P99 ms'].map(num), e=d['P99 ms'].map(sdv)).sort_values('p'); y = np.arange(len(d)); ax.barh(y, d['p'], xerr=d['e'], color=OI['blue'], capsize=2)
    for xv, lab in [(10, 'Tier 1: 10 ms'), (100, 'Tier 2: 100 ms')]: ax.axvline(xv, ls='--', color=OI['verm'], lw=1); ax.text(xv * 1.05, -0.45, lab, fontsize=8, color=OI['verm'], ha='left', va='bottom')
    ax.set_xscale('log'); ax.set_yticks(y); ax.set_yticklabels(d['component'], fontsize=8); ax.set_xlabel('P99 latency per row (ms, log scale)')
@figure('N-MSU1', 'Results: cross-dataset validation (MSU / Morris)', 'Macro-F1 of the default and tuned XGBoost and the Random Forest on the MSU/Morris data for each split and task. Mean and std over seeds.', ['msu_summary'], size=(9.5, 4.5))
def nmsu1(fig, ax):
    d = rd('msu_summary'); d['grp'] = d['split'].astype(str) + '\n' + d['task'].astype(str); grp = list(dict.fromkeys(d['grp'])); mods = list(dict.fromkeys(d['model'])); cols = [OI['grey'], OI['blue'], OI['green'], OI['orange'], OI['verm'], OI['sky'] if 'sky' in OI else '#56B4E9']; ser, er = [], []
    for m in mods:
        s = d[d['model'] == m].set_index('grp'); ser.append([num(s.loc[g, 'macro_f1']) if g in s.index else np.nan for g in grp]); er.append([sdv(s.loc[g, 'macro_f1']) if g in s.index else 0 for g in grp])
    bars(ax, grp, ser, cols[:len(mods)], mods, err=er, fmt='{:.2f}', ymax=1.1); ax.set_ylabel('macro-F1'); ax.tick_params(axis='x', labelsize=7); ax.set_title('xgb_default / xgb_tuned = XGBoost; lr = logistic regression; rf = random forest; mlp = neural net', fontsize=8, loc='left', color=OI['grey'])

## 7. Index and captions

In [ ]:
idx = pd.DataFrame(INDEX); idx.to_csv(f'figure_index_{V}.csv', index=False); idx.to_csv(os.path.join(FIGDIR, f'figure_index_{V}.csv'), index=False)
with open(os.path.join(FIGDIR, f'figure_captions_{V}.md'), 'w', encoding='utf-8') as f:
    f.write('# PhysAR-IDS figures (v6): one caption per file\n\n')
    for _, r in idx.iterrows(): f.write(f'**{r["id"]}**  `{r["file"]}`  -- {r["place in dissertation"]}\n\n{r["caption"]}\n\n*Source tables: {r["sources"]}.*\n\n')
json.dump({'notebook': 'NB15', 'pipeline_version': V, 'matplotlib': matplotlib.__version__, 'figures_ok': int((idx['status'] == 'ok').sum()), 'figures_total': len(idx), 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb15_run_manifest_{V}.json', 'w'), indent=1)
print(idx[['id', 'file', 'place in dissertation', 'status']].to_string(index=False)); bad = idx[idx['status'] != 'ok']
print(f'\n{(idx.status == "ok").sum()} of {len(idx)} figures written to {FIGDIR}/' + ('' if bad.empty else f'; {len(bad)} missing a source table -- see the status column.'))